# 02 · Pipeline RAG passo a passo

O enunciado pede para **demonstrar** cada etapa: transformação da pergunta, recuperação semântica, seleção dos conteúdos mais relevantes, construção do contexto e geração da resposta. Este notebook roda uma pergunta por cada etapa e mostra o que entra e o que sai.

As etapas 1 (reescrita) e 5 (geração) usam o Claude e só rodam com `ANTHROPIC_API_KEY` no `.env`. Sem a chave, o notebook mostra a recuperação, que é local.

In [1]:
import os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)

from src import config
from src.generation import llm
from src.indexing.embeddings import codificar_consultas
from src.retrieval.analise_pergunta import analisar_com_llm, analisar_por_regras
from src.retrieval.busca import buscar, rrf
from src.retrieval.indice import carregar

idx = carregar()
PERGUNTA = "O que os clientes relatam sobre problemas relacionados à entrega?"
print(f"{len(idx.docs):,} documentos | vetores {idx.vetores.shape} | LLM disponível: {llm.disponivel()}")

36,038 documentos | vetores (36038, 768) | LLM disponível: False


## 1. Processamento da pergunta

O gestor pergunta com vocabulário de gestor ("problemas relacionados à entrega"); o cliente escreve "não chegou", "atrasou", "até hoje nada". O LLM decide o escopo, o tipo de pergunta, os filtros e gera reescritas no vocabulário do cliente.

In [2]:
analise = analisar_com_llm(PERGUNTA, idx.categorias) if llm.disponivel() else analisar_por_regras(PERGUNTA)
print("origem:", analise.origem, "| escopo:", analise.no_escopo, "| tipo:", analise.tipo, "| tema:", analise.tipo_tema)
print("filtros:", analise.filtros.descrever())
for c_ in analise.consultas: print(" -", c_)

origem: regras | escopo: True | tipo: exploratoria | tema: problema
filtros: sem filtros
 - O que os clientes relatam sobre problemas relacionados à entrega?
 - problemas relacionados à entrega


## 2. Representação vetorial

A pergunta vira um vetor de 768 dimensões com o mesmo modelo que indexou as avaliações (`multilingual-e5-base`, prefixo `query:`). Como os vetores são normalizados, o produto interno é a similaridade de cosseno.

In [3]:
v = codificar_consultas([PERGUNTA])[0]
sims = idx.vetores @ v
top = np.argsort(-sims)[:8]
pd.DataFrame({"similaridade": sims[top].round(3), "nota": idx.docs.loc[top, "nota"].values, "texto": idx.docs.loc[top, "texto"].values})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,similaridade,nota,texto
0,0.864,1,"Em partes. Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, p..."
1,0.859,1,ainda não recebi o meu produto quando vão me entregar ? ou meu dinheiro de volta
2,0.859,1,TERRIVEL ESTE PROBLEMA DA ENTREGA
3,0.857,4,Boa. Fatisfeito com o produto é a entrega
4,0.855,2,Como podem mandar um produto errado ? Não existe conferências? Loja falha
5,0.855,2,Produto não entregue. Gostaria de informações sobre a não entrega do produto.
6,0.855,1,Como resolver esta situação q o produto ainda não foi entregue?
7,0.854,1,Produto não entregue. O produto ainda não foi entregue. Recebi a mensagem de que houve dificuldade na entrega


## 3. Recuperação híbrida e fusão

Busca densa (sentido) e BM25 (palavras exatas) trazem candidatos diferentes. A Reciprocal Rank Fusion soma `1/(60 + posição)` de cada lista: quem aparece bem nas duas sobe.

In [4]:
permitidos = np.arange(len(idx.docs))
densa = idx.busca_densa(v, permitidos, 100)
bm25 = idx.busca_bm25(PERGUNTA, permitidos, 100)
fundido = rrf([densa, bm25])
so_densa = {p for p, _ in densa} - {p for p, _ in bm25}
so_bm25 = {p for p, _ in bm25} - {p for p, _ in densa}
print(f"densa: {len(densa)} | bm25: {len(bm25)} | nas duas: {len({p for p,_ in densa} & {p for p,_ in bm25})} | após fusão: {len(fundido)}")
print("\nSó o BM25 achou (palavra exata):")
for p in list(so_bm25)[:3]: print(" -", idx.docs.at[p, "texto"][:120])
print("\nSó a busca densa achou (sentido parecido, outras palavras):")
for p in list(so_densa)[:3]: print(" -", idx.docs.at[p, "texto"][:120])

densa: 100 | bm25: 100 | nas duas: 9 | após fusão: 191

Só o BM25 achou (palavra exata):
 - Eu pedi a devolução do dinheiro, após inúmeras tentativas de contato sobre a entrega e nem sobre isso obtive resposta.
 - Compra tranquila e entrega sem problemas.
 - Recebido rapidamente e sem problemas

Só a busca densa achou (sentido parecido, outras palavras):
 - TERRIVEL ESTE PROBLEMA DA ENTREGA
 - Não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz
 - Produto não foi entregue. Gostaria de saber como vai ficar minha situação, pois o produto não foi entague


## 4. Seleção: reranking, limiar e diversidade

O cross-encoder lê pergunta e avaliação juntas e dá um score de relevância. Avaliações abaixo do limiar calibrado (`reports/calibracao_limiar.json`) não contam como evidência; com menos de 3 acima dele, o sistema recusa. O MMR evita 15 variações da mesma frase.

In [5]:
r = buscar(analise.consultas, analise.filtros)
print(f"universo {r.universo:,} | candidatos fundidos {r.candidatos_fundidos} | abaixo do limiar {r.descartadas_limiar} | suficiente: {r.suficiente}")
pd.DataFrame([e.como_dict() for e in r.evidencias])[["score_rerank", "nota", "status_entrega", "categoria", "texto"]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

universo 36,038 | candidatos fundidos 319 | abaixo do limiar 37 | suficiente: True


,score_rerank,nota,status_entrega,categoria,texto
0,2.710,3,atrasado,perfumaria,"Só tive problemas Com relação a entrega, mas agora recebi ; super satisfeita"
1,1.883,1,no_prazo,relogios presentes,muitos problemas na entrega do mesmo. não fui avisada sobre a entrega.
2,1.194,4,no_prazo,telefonia,Produto entregue se problemas obrigado
3,0.797,1,no_prazo,cool stuff,"Em partes. Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, p..."
4,0.718,1,atrasado,informatica acessorios,TERRIVEL ESTE PROBLEMA DA ENTREGA
5,0.686,5,no_prazo,papelaria,Nao tive problemas na entrega nota *****
6,0.542,2,nao_entregue,beleza saude,"Tive problemas com a entrega, não recebi o produto ainda."
7,0.541,1,no_prazo,informatica acessorios,problemas na entrega. infelizmente essa compra não foi bem sucedida
8,0.538,2,atrasado,malas acessorios,tive problemas com a entrega a mercadoria não foi entrgue
9,0.470,4,no_prazo,telefonia fixa,A entrega ocorreu sem problemas. O produto funciona perfeitamente.


## 5. Contexto e geração

O contexto numera as avaliações [R1]..[Rn] com seus metadados. O prompt exige citação em cada afirmação e proíbe números que não venham da tabela de frequência. Depois da geração, o código remove citações para avaliações que não estão no contexto.

In [6]:
from src.generation.resposta import gerar, montar_contexto
print(montar_contexto(PERGUNTA, analise.filtros.descrever(), r.evidencias, None)[:2500])

Pergunta do gestor: O que os clientes relatam sobre problemas relacionados à entrega?

Filtros aplicados: sem filtros

Avaliações recuperadas:
[R1] nota 3 | perfumaria | entregue com atraso | compra 2017-02-28
"Só tive problemas Com relação a entrega, mas agora recebi ; super satisfeita"
[R2] nota 1 | relogios presentes | entregue no prazo | compra 2018-04-27
"muitos problemas na entrega do mesmo. não fui avisada sobre a entrega."
[R3] nota 4 | telefonia | entregue no prazo | compra 2017-09-19
"Produto entregue se problemas obrigado"
[R4] nota 1 | cool stuff | entregue no prazo | compra 2018-06-28
"Em partes. Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, porém foi entregue apenas 1 produto, aguardo solução!"
[R5] nota 1 | informatica acessorios | entregue com atraso | compra 2018-03-22
"TERRIVEL ESTE PROBLEMA DA ENTREGA"
[R6] nota 5 | papelaria | entregue no prazo | compra 2018-03-20
"Nao tive problemas na entrega nota *****"
[R

In [7]:
if llm.disponivel():
    resp = gerar(PERGUNTA, analise.filtros.descrever(), r.evidencias, None)
    print(resp.texto)
    print("\ncitações válidas:", resp.citacoes_validas, "| inválidas removidas:", resp.citacoes_invalidas)
    print("lacunas:", resp.lacunas)
else:
    print("Sem ANTHROPIC_API_KEY: geração não executada. Exemplos gerados em reports/exemplos_respostas.json.")

Sem ANTHROPIC_API_KEY: geração não executada. Exemplos gerados em reports/exemplos_respostas.json.


## 6. Quando o sistema deve dizer que não sabe

In [8]:
from src.pipeline import responder
for q in ["Qual foi o faturamento da Olist em 2017?", "O que os clientes acham das entregas feitas por drone?"]:
    res = responder(q)
    print(f"{res.status:24s} max_score={res.busca.max_score if res.busca else None}  {q}\n   -> {res.mensagem}\n")

evidencia_insuficiente   max_score=-1.9760935306549072  Qual foi o faturamento da Olist em 2017?
   -> Não encontrei avaliações suficientes para responder com segurança (0 avaliações relevantes; o mínimo é 3). Tente ampliar os filtros ou reformular a pergunta.



evidencia_insuficiente   max_score=-1.2357964515686035  O que os clientes acham das entregas feitas por drone?
   -> Não encontrei avaliações suficientes para responder com segurança (1 avaliações relevantes; o mínimo é 3). Tente ampliar os filtros ou reformular a pergunta.

